# Analog chain walkthrough

The chain objects are plain Python, so anything the GUI can do is available here —
and a chain built in either place is the same object saved in the same format.

This notebook covers:

1. Loading a `.json` chain file (including one downloaded from the browser GUI)
2. Inspecting what is in a chain
3. Gain: at a point, between two points, and swept — the browser's left plot
4. Noise: the spectrum and the per-source breakdown — the browser's right plot
5. The noise budget at a reference plane — the browser's table
6. Changing a component and comparing
7. Building a chain from scratch and saving it
8. `chain_api`, the same facade the browser drives

**Two frequencies appear throughout and they are not interchangeable.** The
*carrier* frequency is the RF tone probing the system (MHz–GHz) and sets noise
*levels*; the *spectral* frequency is the offset from that carrier (Hz–MHz) and
sets the noise *shape*. Every `noise()` takes both.

In [ ]:
import os
import sys

# The modules are top-level, so the repo root has to be importable. Skip this
# if you installed the wheel (pip install analog-chain-core).
REPO_ROOT = os.path.dirname(os.path.abspath(""))
if REPO_ROOT not in sys.path:
    sys.path.insert(0, REPO_ROOT)

import matplotlib.pyplot as plt
import numpy as np

from signal_chain import SignalChain
from utils import to_dbm

plt.rcParams.update({"figure.figsize": (7.5, 3.6), "figure.dpi": 110,
                     "axes.grid": True, "grid.alpha": 0.25,
                     "font.size": 9, "legend.fontsize": 8})

CARRIER = 1.5e9   # Hz — the RF tone
SPECTRAL = 1.0e3  # Hz — offset from that tone
print(f"numpy {np.__version__}")

## 1. Load a chain from a `.json` file

This is exactly what the browser GUI's **download chain.json** button produces —
there is no separate export format. Point `CHAIN_FILE` at your own download to
pick up where you left off in the browser.

In [ ]:
CHAIN_FILE = os.path.join(REPO_ROOT, "examples", "simple_cryogenic_system.json")

chain = SignalChain.load(CHAIN_FILE)
chain.summary()

`load` reports anything it had to substitute. A chain file written before a
parameter existed still loads, but the default it falls back to is *announced* —
silent default substitution is the failure mode this format exists to prevent,
so a file is never quietly a different chain than the one that was saved.

In [ ]:
import json

with open(CHAIN_FILE) as fh:
    raw = json.load(fh)

# saved_utc is written into the file by to_dict but not read back by
# from_dict, so it lives on the file rather than on the object.
print(f"file format:  v{raw['format_version']}")
print(f"file saved:   {raw['saved_utc']}")
print(f"name:         {chain.name}")
print(f"description:  {chain.description}")
print(f"components:   {len(chain)}  (+ DAC and ADC)")


## 2. What is in the chain

`stages()` is the full signal path as `(label, component, kind)` — DAC first,
then the components in order, then the ADC. This is the list the browser's
middle column renders.

Note `noise_reference`: it decides whether a component's own gain is applied to
its noise. An amplifier's noise temperature is quoted at its *input*, so it is
amplified along with the signal; an attenuator's Johnson noise appears at its
*output* and is therefore not attenuated by it.

In [ ]:
header = f"{'#':>2}  {'label':<14} {'kind':<8} {'ref':<7} {'class':<22} params"
print(header)
print("-" * len(header))
for i, (label, component, kind) in enumerate(chain.stages()):
    ref = getattr(component, "noise_reference", "—")
    params = ", ".join(f"{k}={v}" for k, v in component.params.items()) or "—"
    print(f"{i:>2}  {label:<14} {kind:<8} {ref:<7} {type(component).__name__:<22} {params}")

Individual components are ordinary objects — call their models directly:

In [ ]:
lna = chain.components[chain.get_index("LNA")]
print(f"{lna.name}: gain {lna.gain(CARRIER):.2f} dB, "
      f"intrinsic noise {to_dbm(lna.noise(CARRIER, SPECTRAL)):.2f} dBm/Hz")

cold = chain.components[chain.get_index("ColdAtten")]
print(f"{cold.name}: gain {cold.gain(CARRIER):.2f} dB, "
      f"temperature {cold.temperature:.1f} K, "
      f"intrinsic noise {to_dbm(cold.noise(CARRIER, SPECTRAL)):.2f} dBm/Hz")

## 3. Gain

`total_gain` covers the whole path including the converters; `gain_between`
takes any two points, addressed by label or index.

In [ ]:
print(f"total roundtrip gain @ {CARRIER/1e9:.2f} GHz:  {chain.total_gain(CARRIER):7.2f} dB")
print(f"input gain through the cold attenuator: "
      f"{chain.gain_between('InputAtten', 'ColdAtten', CARRIER):7.2f} dB")
print(f"input gain to the output of cold LNA: "
      f"{chain.gain_between('InputAtten', 'LNA', CARRIER):7.2f} dB")
print(f"LNA through the output:       "
      f"{chain.gain_between('LNA', 'WarmAmp2', CARRIER):7.2f} dB")

Every model broadcasts over numpy arrays, so a sweep is one call. This is the
browser's **total gain vs carrier frequency** plot.

In [ ]:
carrier_sweep = np.linspace(0.1e9, 3.0e9, 401)
gain_db = chain.total_gain(carrier_sweep)

fig, ax = plt.subplots()
ax.plot(carrier_sweep / 1e9, gain_db, color="#d9a441", lw=1.6)
ax.set_xlabel("carrier frequency (GHz)")
ax.set_ylabel("total gain (dB)")
ax.set_title(f"{chain.name} — total gain")
print(f"{gain_db.min():.2f} dB at {carrier_sweep[gain_db.argmin()]/1e9:.3f} GHz"
      f"  ..  {gain_db.max():.2f} dB at {carrier_sweep[gain_db.argmax()]/1e9:.3f} GHz")
plt.show()

## 4. Noise spectrum

`output_noise` returns the total PSD in W/Hz at the chain output. Passing
`contributions=True` also returns the per-source breakdown, already referred to
the output — which is the browser's **per-source breakdown** toggle.

The sweep is over *spectral* frequency at a fixed carrier.

In [ ]:
spectral_sweep = np.logspace(-2, 3, 201)   # 0.01 Hz to 1 kHz
total_w, contributions = chain.output_noise(
    CARRIER, spectral_sweep, contributions=True)

# contributions is {label: W/Hz}, ordered largest-first by peak.
for label, watts in contributions.items():
    print(f"  {label:<14} peak {to_dbm(np.nanmax(watts)):8.2f} dBm/Hz")

In [ ]:
fig, ax = plt.subplots()
ax.semilogx(spectral_sweep, to_dbm(total_w), color="#d9a441", lw=2.0,
            label="total", zorder=3)
for label, watts in contributions.items():
    ax.semilogx(spectral_sweep, to_dbm(watts), lw=1.0, ls="--", label=label)

ax.set_xlabel("spectral offset from carrier (Hz)")
ax.set_ylabel("noise PSD (dBm/Hz)")
ax.set_title(f"output-referred noise @ {CARRIER/1e9:.2f} GHz carrier")
ax.legend(ncol=2, loc="upper right")
plt.show()

The DAC's 1/f skirt dominates close to the carrier and rolls off, which is why
the total tracks it at low offset and flattens onto the broadband floor further
out.

A gap in the DAC trace is expected: the phase-noise model is a fit over
datasheet points and goes non-finite just outside that range
(`hardware_models.py:137`). `np.nan` plots as a break rather than a wrong value.

## 5. The noise budget at a reference plane

This is the browser's table. Every noise source in the system is referred to one
plane: sources upstream are referred forward, sources downstream are referred
*backward* by dividing out the gain between them.

That backward referral is why the total is **not** a power you could measure at
the plane — an ADC sitting behind an attenuator can dominate the budget at the
input of an LNA. It answers "what limits me here", not "what would a power meter
read here".

`at=` picks which side of the named component.

In [ ]:
budget = chain.noise_budget("LNA", CARRIER, SPECTRAL, at="input")
print(budget.table("dBm/Hz"))

In [ ]:
print(f"referred to:  {budget.reference}")
print(f"total:        {budget.total_dbm_per_hz:.2f} dBm/Hz "
      f"= {budget.total_w:.4e} W/Hz = {budget.total_k:.1f} K")
print(f"dominant:     {budget.dominant().label} "
      f"({budget.fraction(budget.dominant())*100:.1f}% of total)")

`to_rows()` is the export shape — every quantity in W/Hz, dBm/Hz *and* K, so
changing units is picking a column rather than converting. It drops straight
into pandas if you have it.

In [ ]:
rows = budget.to_rows()
print(f"columns: {list(rows[0])}\n")

try:
    import pandas as pd
    display(pd.DataFrame(rows)[[
        "source", "kind", "referred_from", "contribution_dBm_per_hz",
        "referral_gain_dB", "contribution_K", "fraction_of_total"]])
except ImportError:
    for row in rows:
        print(f"  {row['source']:<14} {row['contribution_dBm_per_hz']:9.2f} dBm/Hz"
              f"  referral {row['referral_gain_dB']:7.2f} dB"
              f"  {row['fraction_of_total']*100:5.1f}%")

Compare planes to see where the budget's character changes. Sweeping the
reference plane along the chain is something the GUI does one plane at a time
and a notebook can do all at once:

In [ ]:
print(f"{'plane':<24} {'total dBm/Hz':>13} {'K':>12}   dominant")
print("-" * 68)
for label, _component, _kind in chain.stages():
    for at in ("input", "output"):
        b = chain.noise_budget(label, CARRIER, SPECTRAL, at=at)
        print(f"{label + ' (' + at + ')':<24} {b.total_dbm_per_hz:13.2f} "
              f"{b.total_k:12.1f}   {b.dominant().label}")

## 6. Change a component and compare

Rebuild the component rather than assigning to an attribute. Several models
precompute interpolators from their parameters in `__init__`, so mutating an
attribute afterwards leaves those stale and the gain keeps coming from the old
value. `registry.create` also re-validates.

In [ ]:
import copy

import registry

variant = copy.deepcopy(chain)
variant.name = "ColdAtten -6 dB"
index = variant.get_index("ColdAtten")
variant.components[index] = registry.create(
    "attenuator", {"attenuation": -6.0, "temperature": 4.0}, name="ColdAtten")

for c in (chain, variant):
    b = c.noise_budget("LNA", CARRIER, SPECTRAL, at="input")
    print(f"{c.name:<24} gain {c.total_gain(CARRIER):6.2f} dB   "
          f"budget {b.total_dbm_per_hz:8.2f} dBm/Hz   dominant {b.dominant().label}")

In [ ]:
fig, ax = plt.subplots()
for c, style in ((chain, "-"), (variant, "--")):
    ax.plot(carrier_sweep / 1e9, c.total_gain(carrier_sweep), style, lw=1.5,
            label=c.name)
ax.set_xlabel("carrier frequency (GHz)")
ax.set_ylabel("total gain (dB)")
ax.set_title("14 dB less cold attenuation")
ax.legend()
plt.show()

Validation is declared once, on the parameter spec, so the notebook and both
GUIs reject the same values with the same message:

In [ ]:
try:
    registry.create("attenuator", {"attenuation": +500.0, "temperature": 4.0})
except ValueError as exc:
    print(f"rejected: {exc}")

spec = registry.resolve("attenuator").param("attenuation")
print(f"\n{spec.name}: {spec.minimum} .. {spec.maximum} {spec.unit} "
      f"(default {spec.default})")

## 7. Build a chain from scratch, and save it

Components can be constructed directly from `hardware_models` or by `type_id`
through the registry. The registry route is what saved files use, so it is the
one that survives a class being renamed.

In [ ]:
from hardware_models import ASU_3GHz_LNA, Attenuator, SMA_SS086_cryo

fresh = SignalChain(name="Minimal cryo chain",
                    description="Warm attenuator, cryo cable, cold LNA.")
fresh.add_component(Attenuator(-10, 300), label="InputAtten")
fresh.add_component(SMA_SS086_cryo(0.5, temperature=4), label="CryoCable")
fresh.add_component(ASU_3GHz_LNA(), label="LNA")
fresh.set_digitizer(registry.create("converter.ad9082_dac",
                                    {"carrier_power_dbm": -10.0}),
                    registry.create("converter.ad9082_adc", {}))

print(f"gain {fresh.total_gain(CARRIER):.2f} dB")
print(fresh.noise_budget("LNA", CARRIER, SPECTRAL, at="input").table())

In [ ]:
out_path = os.path.join(REPO_ROOT, "examples", "minimal_cryo_chain.json")
fresh.save(out_path)

# Round-trip check: what came back is the same chain, not an approximation.
reloaded = SignalChain.load(out_path)
assert reloaded.to_dict()["components"] == fresh.to_dict()["components"]
assert reloaded.total_gain(CARRIER) == fresh.total_gain(CARRIER)
print(f"saved and verified: {out_path}")

# Drop this file on the browser GUI's "open chain…" button and it loads there too.
os.remove(out_path)

## 8. `chain_api` — the facade the browser drives

The browser GUI has no chain of its own. It calls `chain_api`, which is in the
same wheel and returns plain JSON-safe dicts. Reaching for it from a notebook is
occasionally handy — it gives the whole component catalog as data, and it
returns errors instead of raising, which suits scripted sweeps.

For interactive work prefer the objects above; `chain_api` holds one
module-level chain, which is convenient for a GUI and awkward for a notebook.

In [ ]:
import chain_api

# The component library, exactly as the browser's left column renders it.
for group in chain_api.catalog()["categories"]:
    names = [c["label"] for c in group["components"]]
    print(f"{group['category']:<12} {len(names):>2}  {', '.join(names[:3])}"
          f"{' …' if len(names) > 3 else ''}")

In [ ]:
chain_api.load_preset("cryo_example")
b = chain_api.budget("LNA", at="input", carrier_hz=CARRIER, spectral_hz=SPECTRAL)
print(f"ok={b['ok']}  total {b['total_dbm_per_hz']:.2f} dBm/Hz  "
      f"dominant {b['dominant']}")

# Failures come back as data rather than raising — the browser needs a message
# it can display, since it cannot inspect a Python traceback.
bad = chain_api.budget("NoSuchPlane", at="input", carrier_hz=CARRIER,
                       spectral_hz=SPECTRAL)
print(f"\nok={bad['ok']}\n{bad['error'][:100]}")

---

## Where to go next

- `chain.stages()` / `resolve_plane()` — how planes are numbered, if you are
  writing your own referral
- `registry.entries()` — every component with its parameter specification
- `hardware_models.py` — the models themselves; each docstring records where its
  numbers come from
- `web/README.md` — the browser build, and the one place its numbers differ
  (a scipy `curve_fit` in the DAC model converges slightly differently between
  scipy versions, so the shared wheel guarantees the same code, not the same
  dependency versions)